## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# Stop-Loss (SL & SL-M)

Welcome! In the earlier sections, we learned how to place basic orders. We now move to the most critical aspect of trading: **risk management**.

A Stop-Loss order is your primary tool for managing risk. It's an instruction to automatically exit a trade if the price moves against you to a predetermined level, preventing a small loss from becoming a large one. In this notebook, you will learn:

- What a Stop-Loss (SL) order is and why it's essential.
- The critical difference between a **Stop-Loss Limit (SL)** and a **Stop-Loss Market (SL-M)** order.
- How to programmatically place both order types using the Kite Connect API.

### 1. Authentication

As always, we first need to authenticate our Kite Connect client to interact with our trading account. Please replace the placeholder credentials with your own.

In [ ]:
from kiteconnect import KiteConnect
import logging

logging.basicConfig(level=logging.INFO)

# --- Credentials & Initialisation ---
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"

try:
    kite = KiteConnect(api_key=api_key)
    kite.set_access_token(access_token)
    profile = kite.profile()
    print(f"Logged in as {profile['user_name']}")
    logging.info("Authentication successful!")
except Exception as e:
    logging.error(f"Authentication failed: {e}")

INFO:root:Authentication successful!


### 2. The Concept: SL vs. SL-M

Zerodha provides two types of stop-loss orders.

#### Stop-Loss Market (SL-M)
This is the simpler of the two. It has only one price:
-   **Trigger Price**: The price at which your exit order is sent to the exchange. The order is a **Market Order**, meaning it will execute at the next available price.
    -   **Pro**: Guaranteed execution. Your position will be closed once the trigger price is hit.
    -   **Con**: Prone to slippage. In a volatile market, the execution price might be worse than your trigger price.

#### Stop-Loss Limit (SL)
This order type has two prices:
-   **Trigger Price**: Same as before, this is the price that activates your exit order.
-   **Limit Price**: This is the worst price you are willing to accept. Your order becomes a **Limit Order** that will only execute at your limit price or better.
    -   **Pro**: You have control over the execution price, protecting you from slippage.
    -   **Con**: Execution is not guaranteed. If the market moves too quickly and gaps past your limit price, your order may never be filled.

### 3. Placing a Stop-Loss Market (SL-M) Order

Here, we're looking at a common scenario: we already have an open **BUY** position in a stock, and now we want to place a stop-loss order to protect our profits or limit potential losses. This code places a **Stop-Loss Market (SL-M)** order, which is a popular way to do this.

Just like before, we're using a `try...except` block to safely place the order and catch any potential errors from the API.

### A Look Inside the Order Parameters

Let's break down what each part of this `kite.place_order` call means:

* **`variety=kite.VARIETY_REGULAR`**: We're placing a regular, standalone order, not a special variety like a Cover Order (CO) or Iceberg.

* **`tradingsymbol='INFY'`** & **`exchange=kite.EXCHANGE_NSE`**: We're working with Infosys shares on the National Stock Exchange.

* **`transaction_type=kite.TRANSACTION_TYPE_SELL`**: This is important! Since we are trying to exit an existing **BUY** position, our stop-loss order needs to be a **SELL** order.

* **`order_type=kite.ORDER_TYPE_SLM`**: This tells the system we want to place a **Stop-Loss Market** order. This means once the `trigger_price` is hit, the system will place a `SELL` order at the current market price.

* **`quantity=10`**: We're setting this stop-loss for 10 shares, which should match the quantity of our open position.

* **`trigger_price=1540.0`**: This is the core of our stop-loss. If the price of INFY drops down to ₹1540.0, our stop-loss order will be "triggered" or activated.

* **`price=0`**: For a Stop-Loss **Market** (SL-M) order, the price must be set to 0. This tells the exchange: "When the trigger price is hit, sell my shares at whatever the best available market price is at that moment."

* **`product=kite.PRODUCT_MIS`**: We're specifying this is for an intraday position.

### What Happens When We Run This?

* **Success**: If everything is correct, the stop-loss order will be placed and sit pending in the order book. We'll get a confirmation message with the `order_id`.
* **Failure**: If there's an issue (like the trigger price being too close to the current market price), the `except` block will catch it and let us know what went wrong.

In [ ]:
# Scenario: We have a BUY position and want to place a stop-loss to sell.

try:
    order_id = kite.place_order(

        # Order Variety is set to regular
        variety=kite.VARIETY_REGULAR,

        # Symbol and Exchange
        tradingsymbol='INFY',
        exchange=kite.EXCHANGE_NSE,

        # We are exiting our BUY position by placing a SELL order
        transaction_type=kite.TRANSACTION_TYPE_SELL, # We are selling to exit our buy position
        order_type=kite.ORDER_TYPE_SLM,              # Stop-Loss Market order

        # Quantity of shares to sell and SL trigger price
        quantity=10,
        trigger_price=1540.0,                        # If INFY hits this price...
        price=0,                                     # For SL-M, price must be 0

        # Product Type (Intraday)
        product=kite.PRODUCT_MIS

    )
    logging.info(f"SL-M order placed successfully. Order ID: {order_id}")
except Exception as e:
    logging.error(f"Error placing SL-M order: {e}")

INFO:root:SL-M order placed successfully. Order ID: 250717200339368


### 4. Placing a Stop-Loss Limit (SL) Order

Now, let's place an SL (Limit) order for the same scenario.

### The Key Change: From a Market Exit to a Limit Exit

The crucial difference in this code is the switch from a Stop-Loss Market (SL-M) order to a **Stop-Loss Limit (SL)** order. This gives you more control over the exit price.

Let's look at the two parameters that changed:

1.  **`order_type=kite.ORDER_TYPE_SL`**: We are now explicitly defining the order as a **Stop-Loss Limit** type.

2.  **`price=1539.50`**: Unlike the SL-M order where the price was `0`, here we set a specific **limit price**.

### What This Means in Practice

When the `trigger_price` of **₹1540.0** is hit, the system doesn't just sell at any market price. Instead, it places a **Limit Sell Order** at **₹1539.50**.

This means you are instructing the broker: "Once the price touches ₹1540.0, I want to sell, but I will not accept any price below ₹1539.50."

This protects you from selling at a much lower price if the market suddenly gaps down (slippage), but it comes with a trade-off: if the market falls below ₹1539.50 too quickly, your order may not get filled, and you would remain in the position.

In [ ]:
# Scenario: We have a BUY position and want to place a stop-loss with a limit price.

try:
    order_id = kite.place_order(

        # Order Variety is set to regular
        variety=kite.VARIETY_REGULAR,

        # Symbol and Exchange
        tradingsymbol='INFY',
        exchange=kite.EXCHANGE_NSE,

        # We are exiting our BUY position by placing a SELL order
        transaction_type=kite.TRANSACTION_TYPE_SELL, # We are selling to exit our buy position
        order_type=kite.ORDER_TYPE_SL,              # Stop-Loss Limit order

        # Quantity of shares to sell and SL trigger price
        quantity=10,
        trigger_price=1540.0,                        # If INFY hits this price...trigger the SL order
        price=1539.50,                               # The limit price for the SL order

        # Product Type (Intraday)
        product=kite.PRODUCT_MIS

    )
    logging.info(f"SL order placed successfully. Order ID: {order_id}")
except Exception as e:
    logging.error(f"Error placing SL order: {e}")

INFO:root:SL order placed successfully. Order ID: 250717200352457


### 5. Conclusion

You now know how to place the two types of stop-loss orders.

-   **SL-M** offers a **guaranteed exit** but may have slippage.
-   **SL** offers **price control** but execution is not guaranteed.

In the next section, we will explore the creation of a low frequency trading system.<br><br>